In [14]:
#!pip install pytorch-tabnet -q
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from pytorch_tabnet.tab_model import TabNetClassifier
from torch.utils.data.sampler import WeightedRandomSampler
import warnings

warnings.filterwarnings("ignore")

In [18]:
FILE_PATH = 'data/diabetes_prediction_dataset.csv'
data = pd.read_csv(FILE_PATH)
target = 'diabetes'

categorical_cols = ['gender', 'smoking_history', 'hypertension', 'heart_disease']

numerical_cols = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

features = categorical_cols + numerical_cols

data['smoking_history'].fillna('No Info', inplace=True)

for col in numerical_cols:
    data[col].fillna(data[col].median(), inplace=True)

cat_idxs = []
cat_dims = []
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    encoders[col] = le

    # lấy index của cột trong danh sách 'features'
    cat_idxs.append(features.index(col))
    # lấy số lượng hạng mục (lớp)
    cat_dims.append(len(le.classes_))


In [19]:
X = data[features].values
y = data[target].values
# Train + Valid 80, Test 20
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train 64, Valid 16, Test 20
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.2, random_state=42, stratify=y_train_val
)

print(f"Train: {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test: {X_test.shape}")

Train: (64000, 8)
Validation: (16000, 8)
Test: (20000, 8)


In [20]:
clf = TabNetClassifier(
    cat_idxs=cat_idxs,
    cat_dims=cat_dims,
    cat_emb_dim=2,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='sparsemax',
    device_name='cuda',
    verbose=1
)

clf.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_val, y_val)],
    eval_name=['validation'],
    eval_metric=['accuracy', 'auc'],
    max_epochs=100,
    patience=20,
    batch_size=2048,
)

epoch 0  | loss: 0.20273 | validation_accuracy: 0.19288 | validation_auc: 0.6264  |  0:00:02s
epoch 1  | loss: 0.12488 | validation_accuracy: 0.17462 | validation_auc: 0.71773 |  0:00:05s
epoch 2  | loss: 0.1107  | validation_accuracy: 0.26712 | validation_auc: 0.86499 |  0:00:08s
epoch 3  | loss: 0.10434 | validation_accuracy: 0.4835  | validation_auc: 0.84946 |  0:00:10s
epoch 4  | loss: 0.10288 | validation_accuracy: 0.58244 | validation_auc: 0.88326 |  0:00:12s
epoch 5  | loss: 0.09735 | validation_accuracy: 0.75331 | validation_auc: 0.88222 |  0:00:15s
epoch 6  | loss: 0.09443 | validation_accuracy: 0.78494 | validation_auc: 0.90195 |  0:00:18s
epoch 7  | loss: 0.0929  | validation_accuracy: 0.86031 | validation_auc: 0.94103 |  0:00:20s
epoch 8  | loss: 0.09774 | validation_accuracy: 0.93188 | validation_auc: 0.95713 |  0:00:22s
epoch 9  | loss: 0.09103 | validation_accuracy: 0.96419 | validation_auc: 0.9703  |  0:00:25s
epoch 10 | loss: 0.08889 | validation_accuracy: 0.90088 | va

In [21]:
preds = clf.predict(X_test)

print(f"Model Accuracy:  {accuracy_score(y_test, preds)}")
print("\nClassification Report (Tập Test):")
print(classification_report(y_test, preds))

Model Accuracy:  0.9718

Classification Report (Tập Test):
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     18300
           1       0.98      0.68      0.80      1700

    accuracy                           0.97     20000
   macro avg       0.98      0.84      0.89     20000
weighted avg       0.97      0.97      0.97     20000

